In [1]:
%pip install requests python-dotenv
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-0"

def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text}
    
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text}

    messages.append(assistant_message)



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [5]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")
stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01Xd2W5W61kDN18yfJDHxALj', content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='F', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='akeUser', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='DB is a sim', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='ulated customer', type='text_d

In [9]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # print(text, end='')
        pass
stream.get_final_message()

Message(id='msg_017pLR1i5JMX22JytmAeJB3n', content=[TextBlock(citations=None, text='The ChronoLex Database contains over 2.3 million timestamped legal precedents from parallel universe court systems, allowing attorneys to search for case law across multiple dimensional timelines.', type='text')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=18, output_tokens=43, server_tool_use=None, service_tier='standard'))

In [13]:
messages = []
add_user_message(messages, "Is tea or coffee better at breakfast?")
add_assistant_message(messages, "Tea is better at breakfast because")
answer = chat(messages)
answer

' it gives you a gentler caffeine boost than coffee, making it easier to ease into your day. Tea also contains L-theanine, an amino acid that promotes calm alertness and can help balance caffeine\'s stimulating effects.\n\nHowever, this really comes down to personal preference! Some people prefer coffee\'s stronger caffeine kick to fully wake up, while others find tea\'s milder effect more pleasant in the morning. Coffee also pairs well with many breakfast foods.\n\nConsider factors like:\n- Your caffeine sensitivity\n- How quickly you want to feel alert\n- What flavors you enjoy with breakfast\n- Any stomach sensitivity (tea is generally gentler)\n\nBoth can be part of a healthy morning routine - the "better" choice is whatever works best for your body and preferences.'

In [4]:
messages = []
add_user_message(messages, "Count from 1 to 10")
answer = chat(messages, stop_sequences=[", 7"])
answer

'1, 2, 3, 4, 5, 6'

In [6]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages,"```json")
text = chat(messages,stop_sequences=["```"])
text

'\n{\n  "Name": "OrderProcessingRule",\n  "EventPattern": {\n    "source": ["myapp.orders"],\n    "detail-type": ["Order Placed"]\n  },\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n'

In [7]:
import json
json.loads(text.strip())

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}

In [8]:
messages = []
prompt = """
Generate three different sample AWS CLI commands. Each should be very short."""
add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")
text = chat(messages, stop_sequences=["```"])
text.strip()

'aws s3 ls\n\naws ec2 describe-instances\n\naws iam list-users'

In [9]:
from IPython.display import Markdown, display
display(Markdown(text.strip()))

aws s3 ls

aws ec2 describe-instances

aws iam list-users